[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/agent-memory.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239417-lesson-7-agent-with-memory)

# Agent 记忆

## 回顾

之前，我们构建了一个 Agent，它可以：

* `act` —— 让模型调用特定工具
* `observe` —— 将工具输出传回给模型
* `reason` —— 让模型根据工具输出推理下一步该做什么（例如，调用另一个工具或直接响应）

![Screenshot 2024-08-21 at 12.45.32 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbab7453080e6802cd1703_agent-memory1.png)

## 目标

现在，我们将通过引入记忆来扩展我们的 Agent。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_deepseek langchain_core langgraph langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

我们将使用 [LangSmith](https://docs.langchain.com/langsmith/home) 进行[追踪](https://docs.langchain.com/langsmith/observability-concepts)。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

这与我们之前的操作相同。

In [ ]:
from langchain_deepseek import ChatDeepSeek

def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

# 这将是一个工具
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a and b.

    Args:
        a: first int
        b: second int
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatDeepSeek(model="deepseek-chat")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# 节点
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些节点执行实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些决定控制流的走向
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果 assistant 的最新消息（结果）是工具调用 -> tools_condition 路由到 tools
    # 如果 assistant 的最新消息（结果）不是工具调用 -> tools_condition 路由到 END
    tools_condition,
)
builder.add_edge("tools", "assistant")
react_graph = builder.compile()

# 显示
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

## Memory（记忆）

让我们和之前一样运行 Agent。

In [ ]:
messages = [HumanMessage(content="Add 3 and 4.")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

现在，让我们将结果乘以 2！

In [ ]:
messages = [HumanMessage(content="Multiply that by 2.")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

我们没有保留初始对话中结果 7 的记忆！

这是因为[状态是临时的](https://github.com/langchain-ai/langgraph/discussions/352#discussioncomment-9291220)，仅限于单次图执行。

当然，这限制了我们进行带中断的多轮对话的能力。

我们可以使用[持久化](https://docs.langchain.com/oss/python/langgraph/persistence)来解决这个问题！

LangGraph 可以使用 Checkpointer（检查点记录器）在每一步之后自动保存图状态。

这个内置的持久化层赋予了我们记忆能力，使得 LangGraph 可以从上一次状态更新处继续。

最简单的 Checkpointer 之一是 `MemorySaver`，它是一个用于图状态的内存键值存储。

我们只需要在编译图时传入一个 Checkpointer，图就拥有了记忆！

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
react_graph_memory = builder.compile(checkpointer=memory)

当我们使用记忆时，需要指定一个 `thread_id`。

这个 `thread_id` 将存储我们的图状态集合。

以下是一个示意图：

* Checkpointer 在图中的每一步写入状态
* 这些检查点保存在一个 thread（线程）中
* 我们可以在将来通过 `thread_id` 访问该线程

![state.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e0e9f526b41a4ed9e2d28b_agent-memory2.png)


In [ ]:
# 指定一个线程
config = {"configurable": {"thread_id": "1"}}

# 指定输入
messages = [HumanMessage(content="Add 3 and 4.")]

# 运行
messages = react_graph_memory.invoke({"messages": messages},config)
for m in messages['messages']:
    m.pretty_print()

如果我们传入相同的 `thread_id`，那么就可以从之前记录的状态检查点继续执行！

在这个例子中，上述对话被捕获在该线程中。

我们传入的 `HumanMessage`（`"Multiply that by 2."`）被追加到上面的对话中。

因此，模型现在知道 `that` 指的是 `The sum of 3 and 4 is 7.`。

In [ ]:
messages = [HumanMessage(content="Multiply that by 2.")]
messages = react_graph_memory.invoke({"messages": messages}, config)
for m in messages['messages']:
    m.pretty_print()

## Studio

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，使其现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而不是使用视频中展示的桌面应用。它现在被称为 *LangSmith Studio* 而不是 *LangGraph Studio*。详细的设置说明可在课程开头的"Getting Setup"指南中找到。你可以[在此](https://docs.langchain.com/langsmith/studio)查看 Studio 的说明，以及[在此](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)查看本地部署的具体细节。

要在本模块的 `/studio` 目录中启动本地开发服务器，请在终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。

在 Studio 中加载 `agent`，它使用 `module-1/studio/langgraph.json` 中设置的 `module-1/studio/agent.py`。